# Broadcasting

Section 13.3 of the notes.

Broadcasting lets an operation combine arrays of different shapes, such as a
matrix and a vector, without copying the smaller one. NumPy decides whether two
shapes fit by comparing them axis by axis, starting from the last axis:

1. two axes fit if they have the same length, or if one of them has length 1;
2. an axis of length 1 is stretched to the length of the other, by reusing its
   single entry;
3. if one array has fewer axes, axes of length 1 are added on its left.

The stretching allocates no memory: NumPy reads the same entry again by giving
that axis the stride $0$ (Section 13.1).

| shape of A | shape of B | result | why |
|---|---|---|---|
| (5, 3) | (3,) | (5, 3) | B becomes (1, 3), and its one row is reused for all 5 rows |
| (5, 3) | (5, 1) | (5, 3) | the single column of B is reused for all 3 columns |
| (4, 3, 2) | (3, 1) | (4, 3, 2) | B becomes (1, 3, 1); its length-1 axes are reused |
| (5, 3) | (5,) | error | the last axes, 3 and 5, differ and neither is 1 |

The next cell shows the stride $0$. `np.broadcast_to` does by itself what an
operation does when it broadcasts: it stretches the first column of `X`, shape
`(3, 1)`, to shape `(3, 2)`. Both columns of the result are the same memory,
read twice, because the step from one column to the next is $0$ bytes.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
X = np.arange(6.0).reshape(3, 2)
print("X:\n", X)

col = X[:, :1]  # the first column of X, shape (3, 1)
B = np.broadcast_to(col, (3, 2))  # the same column, stretched to two columns

print("\nfirst column, shape", col.shape, ":\n", col)

print("\nstretched, shape", B.shape, ":\n", B)

print("\nstrides of the column: ", col.strides)  # 16 bytes to the next row
print("strides when stretched:", B.strides)  # 0 bytes to the next column
print("B shares memory with X:", np.shares_memory(B, X))

## 1. Centering rows and columns

Subtracting the mean of each column, or of each row, is the most common use.
`X.mean(axis=0)` has shape `(2,)`, one mean per column; by rule 3 it becomes
`(1, 2)` and is reused for every row. `X.mean(axis=1)` has shape `(3,)`, one
mean per row, and it does not fit `(3, 2)`, because shapes are compared from
the right. `keepdims=True` keeps the reduced axis with length 1, shape `(3, 1)`,
which is reused for every column.

In [ ]:
col_mean = X.mean(axis=0)  # shape (2,)
row_mean = X.mean(axis=1, keepdims=True)  # shape (3, 1)


print("\ncol_mean:\n", col_mean)
print("X - col_mean:\n", X - col_mean)
print()

print("row_mean:\n", row_mean)
print("X - row_mean:\n", X - row_mean)

try:
    X - X.mean(axis=1)  # shape (3,) does not fit the last axis
except ValueError as err:
    print("\nwithout keepdims:", err)

## 2. All pairs of distances

The rows of $\mathbf{X}$, of shape $(n, d)$, are $n$ points $x_1,\dots,x_n$ in
$d$ dimensions, and the rows of $\mathbf{Y}$, of shape $(m, d)$, are $m$ further
points $y_1,\dots,y_m$. We want the $n\times m$ matrix of all their distances,
$D_{ij} = \|x_i - y_j\|$.

Broadcasting gives all the differences $x_i - y_j$ at once:

1. `X[:, None, :]` inserts an axis of length 1 in the middle: shape $(n, 1, d)$.
2. `Y[None, :, :]` inserts one in front: shape $(1, m, d)$.
3. Their difference has shape $(n, m, d)$, by the rules above: the length-1
   axis of `X` is reused for all $m$ points of $\mathbf{Y}$, and that of `Y`
   for all $n$ points of $\mathbf{X}$. Its entry `[i, j, :]` is the vector
   $x_i - y_j$.
4. Squaring and summing over the last axis gives $\|x_i - y_j\|^2$, and the
   square root gives $D_{ij}$.

The code is one line, but step 3 stores every difference vector before step 4
reduces it to one number: $n\cdot m\cdot d$ numbers on the way to a result of
$n\cdot m$ numbers, $d$ times more memory than the answer needs.

How large is that? The next cell does not compute any distances. It only
counts the memory for $n=4000$ points against $m=3000$ points in $d=64$
dimensions: each entry is a float64 of 8 bytes, so an array needs 8 bytes times
its number of entries. The result has $nm$ entries, the intermediate array
$nmd$.

In [ ]:
n, m, d = 4000, 3000, 64
# memory = 8 bytes per float64 entry x number of entries
print(f"result D, shape (n, m):        {n * m * 8 / 1e6:8.1f} MB")
print(f"difference, shape (n, m, d):   {n * m * d * 8 / 1e9:8.1f} GB")

Expanding the square,

$$ \|x - y\|^2 = \|x\|^2 + \|y\|^2 - 2\,x^\top y , $$

gives the same matrix from the squared lengths of the rows and one matrix
product, `X @ Y.T`, with no array larger than the result. In floating point the
right side can be slightly negative for nearly equal points, which is why the
code clips it at $0$ before the square root.

In [ ]:
def distances_broadcast(X, Y):
    return np.sqrt(((X[:, None, :] - Y[None, :, :]) ** 2).sum(axis=2))


def distances_matmul(X, Y):
    d2 = (X * X).sum(1)[:, None] + (Y * Y).sum(1)[None, :] - 2 * (X @ Y.T)
    return np.sqrt(np.maximum(d2, 0))  # rounding can give small negatives


Xs, Ys = rng.standard_normal((300, d)), rng.standard_normal((200, d))
print(
    "same result:", np.allclose(distances_broadcast(Xs, Ys), distances_matmul(Xs, Ys))
)

print("\nsmall problem, 300 x 200:")
%timeit -r 3 distances_broadcast(Xs, Ys)
%timeit -r 3 distances_matmul(Xs, Ys)

In [ ]:
# at the full size only the second version is possible on a normal machine
Xb, Yb = rng.standard_normal((n, d)), rng.standard_normal((m, d))
%timeit -r 1 -n 1 distances_matmul(Xb, Yb)

## 3. When a matrix product is faster

Broadcasting removes the Python loop, but the computation may still build a
large intermediate array. The example is the weighted sum of each row of a
matrix $A$ of shape $(2000, 64)$ with a weight vector $w$ of length $64$,
$$ r_i = \sum_{j} A_{ij}\,w_j , \qquad i = 1,\dots,2000 . $$
It can be written in three ways, which give the same numbers:

1. `(A * w).sum(axis=1)` broadcasts `w` against every row, which builds the
   whole array of products $A_{ij}w_j$, as large as $A$, then sums each row.
   The products are written to memory and read back.
2. `A @ w` is a matrix-vector product. NumPy hands it to an optimised linear
   algebra library, which multiplies and adds in one pass over $A$, without
   storing the products.
3. `np.einsum("ij,j->i", A, w)` also builds no intermediate array, but it runs
   NumPy's general loop instead of the optimised library. How to read the
   string is explained at the end of this section.

In [ ]:
A = rng.standard_normal((2000, 64))
w = rng.standard_normal(64)

print("weighted row sums, three ways, same result:")
r1 = (A * w).sum(axis=1)  # temporary of the size of A
r2 = A @ w  # one matrix-vector product
r3 = np.einsum("ij,j->i", A, w)  # no temporary
print(np.allclose(r1, r2), np.allclose(r1, r3))
%timeit -r 3 (A * w).sum(axis=1)
%timeit -r 3 A @ w
%timeit -r 3 np.einsum("ij,j->i", A, w)

The times follow that order: the broadcast version is the slowest, because it
moves the extra array through memory, `einsum` is faster, and `A @ w` is the
fastest. When an expression is a sum of products, as here, write it as a matrix
product if you can. Before writing any expression, multiply out the shape of
each intermediate array to see how much memory it needs.

**Reading an `einsum` string.** The string describes a loop. Before `->` it
gives the indices of each input, one group of letters per array; after `->`
it gives the indices of the result. `einsum` multiplies the indexed entries and
adds over every index that does not appear after `->`. Three examples, each
with the loop it stands for.

**1. Weighted row sums,** $r_i=\sum_j A_{ij}w_j$, string `"ij,j->i"`:

- `ij` : the first array, `A`, is indexed as `A[i, j]`;
- `j`  : the second array, `w`, is indexed as `w[j]`;
- `->i`: the result is indexed as `r[i]`;
- summed: `j`, since it is missing after `->`.

```python
r = np.zeros(A.shape[0])
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        r[i] += A[i, j] * w[j]
```

**2. Matrix product,** $P_{ik}=\sum_j A_{ij}B_{jk}$, string `"ij,jk->ik"`, the
same as `A @ B`:

- `ij`  : the first array, `A`, is indexed as `A[i, j]`;
- `jk`  : the second array, `B`, is indexed as `B[j, k]`;
- `->ik`: the result is indexed as `P[i, k]`;
- summed: `j`.

```python
P = np.zeros((A.shape[0], B.shape[1]))
for i in range(A.shape[0]):
    for k in range(B.shape[1]):
        for j in range(A.shape[1]):
            P[i, k] += A[i, j] * B[j, k]
```

**3. Trace,** $t=\sum_i S_{ii}$, string `"ii->"`:

- `ii`: the one array, `S`, is indexed as `S[i, i]`, the same letter twice, so
  only its diagonal entries are used;
- `->` with nothing after it: the result is a single number `t`;
- summed: `i`.

```python
t = 0.0
for i in range(S.shape[0]):
    t += S[i, i]
```

The next cell runs these loops on small arrays and compares them with `einsum`.

In [ ]:
A_s = np.arange(6.0).reshape(2, 3)  # entries A[i, j], i = 0..1, j = 0..2
w_s = np.array([1.0, 10.0, 100.0])  # entries w[j]
B_s = np.arange(12.0).reshape(3, 4)  # entries B[j, k]
S = np.arange(9.0).reshape(3, 3)  # entries S[i, i] on the diagonal

r = np.zeros(2)  # "ij,j->i"
for i in range(2):
    for j in range(3):
        r[i] += A_s[i, j] * w_s[j]
print('"ij,j->i"   loop', r, " einsum", np.einsum("ij,j->i", A_s, w_s))

P = np.zeros((2, 4))  # "ij,jk->ik"
for i in range(2):
    for k in range(4):
        for j in range(3):
            P[i, k] += A_s[i, j] * B_s[j, k]
print(
    '"ij,jk->ik" loop and einsum agree:',
    np.allclose(P, np.einsum("ij,jk->ik", A_s, B_s)),
)

t = 0.0  # "ii->"
for i in range(3):
    t += S[i, i]
print('"ii->"      loop', t, " einsum", np.einsum("ii->", S))